# Steam Games – Data Mining Project
## Section 3: Exploratory Data Analysis

**Team 9 – Brewed Clusters**  
Owner: Arpitha Shivaprasad Kori

---

### What does this notebook do?

This notebook performs exploratory data analysis on the preprocessed dataset produced by Section 2.  
It does **not** modify the data — the goal is to understand patterns and inform modelling decisions.

Analyses covered:
1. Class balance — how many Good vs Bad games?
2. Review ratio distribution — where does the 0.70 threshold sit?
3. Price distribution — free-to-play vs paid games
4. Top genres and their relationship to reception
5. Top tags and their relationship to reception
6. Platform availability (Windows / Mac / Linux)
7. Numeric feature distributions (playtime, achievements, DLC count)
8. Correlation heatmap of continuous features
9. Outlier identification (IQR method)
10. Feature-label relationships for the most informative features

---
## Step 0 – Import Libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import json

pd.set_option('display.max_columns', 50)
sns.set_theme(style='whitegrid', palette='muted')

GOOD_COLOR = 'steelblue'
BAD_COLOR  = 'salmon'

print('Libraries loaded successfully!')

---
## Step 1 – Load Preprocessed Data

We load directly from the Parquet files produced by `run_preprocessing.py`.  
**Do not re-run preprocessing** — just load what is already saved.

In [ ]:
X = pd.read_csv('Data/processed/X_classification.csv')
y = pd.read_csv('Data/processed/y_classification.csv')['label']

continuous_cols = [
    'log_price', 'Required age', 'DiscountDLC count', 'Achievements',
    'Average playtime forever', 'Median playtime forever',
    'Recommendations', 'Metacritic score', 'n_languages'
]

print(f'Feature matrix : {X.shape[0]:,} rows x {X.shape[1]} columns')
print(f'Labels         : {y.shape[0]:,} entries')
print(f'Continuous cols: {continuous_cols}')

---
## Step 2 – Class Balance

The label is binary: **1 = Good** (>=70% positive reviews), **0 = Bad** (<70%).  
Understanding the imbalance ratio is critical — it directly affects which evaluation metrics to use.

In [ ]:
counts = y.value_counts().sort_index()
pcts   = y.value_counts(normalize=True).sort_index() * 100

print('Class distribution:')
print(f'  Good (1) : {counts[1]:>7,}  ({pcts[1]:.1f}%)')
print(f'  Bad  (0) : {counts[0]:>7,}  ({pcts[0]:.1f}%)')
print(f'  Imbalance ratio (Good:Bad) approx {counts[1]/counts[0]:.1f}:1')

fig, ax = plt.subplots(figsize=(5, 4))
bars = ax.bar(['Bad (<70%)', 'Good (>=70%)'], [counts[0], counts[1]],
               color=[BAD_COLOR, GOOD_COLOR], edgecolor='white', linewidth=0.8)
for bar, val, pct in zip(bars, [counts[0], counts[1]], [pcts[0], pcts[1]]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
            f'{val:,}\n({pct:.1f}%)', ha='center', va='bottom', fontsize=10)
ax.set_title('Class Distribution', fontsize=13, fontweight='bold')
ax.set_ylabel('Number of Games')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
plt.tight_layout()
plt.show()

---
## Step 3 – Price Distribution

We analyse `log_price` (the log-transformed price stored in X) and also recover the approximate original price.  
The key question: do free-to-play games tend to receive better or worse reviews?

In [ ]:
# Recover approximate original price from log_price
# log_price = log1p(price)  ->  price = exp(log_price) - 1
price_approx = np.expm1(X['log_price'])
is_free      = (price_approx == 0)

print(f'Free-to-play games : {is_free.sum():,} ({is_free.mean()*100:.1f}%)')
print(f'Paid games         : {(~is_free).sum():,} ({(~is_free).mean()*100:.1f}%)')
print(f'Paid price -- median: ${price_approx[~is_free].median():.2f}  max: ${price_approx.max():.2f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: log_price distribution by class
for label, color, name in [(0, BAD_COLOR, 'Bad'), (1, GOOD_COLOR, 'Good')]:
    axes[0].hist(X.loc[y == label, 'log_price'], bins=40, alpha=0.6,
                 color=color, label=name, density=True)
axes[0].set_title('Log-Price Distribution by Class')
axes[0].set_xlabel('log_price  [= log(1 + price)]')
axes[0].set_ylabel('Density')
axes[0].legend()

# Right: Good/Bad split among free vs paid
free_good = ((is_free) & (y == 1)).sum()
free_bad  = ((is_free) & (y == 0)).sum()
paid_good = ((~is_free) & (y == 1)).sum()
paid_bad  = ((~is_free) & (y == 0)).sum()

categories = ['Free-to-Play', 'Paid']
good_vals  = [free_good / (free_good + free_bad) * 100,
              paid_good / (paid_good + paid_bad) * 100]
bad_vals   = [100 - g for g in good_vals]

x_pos = np.arange(len(categories))
axes[1].bar(x_pos, good_vals, label='Good', color=GOOD_COLOR, alpha=0.8)
axes[1].bar(x_pos, bad_vals, bottom=good_vals, label='Bad', color=BAD_COLOR, alpha=0.8)
axes[1].axhline(70, color='black', linestyle='--', linewidth=0.8, label='70% threshold')
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(categories)
axes[1].set_ylabel('% of Games')
axes[1].set_title('Good vs Bad by Price Category')
axes[1].legend()

for i, (g, b) in enumerate(zip(good_vals, bad_vals)):
    axes[1].text(i, g/2, f'{g:.1f}%', ha='center', va='center', fontsize=9, color='white', fontweight='bold')

plt.tight_layout()
plt.show()

---
## Step 4 – Top Genres

Genres are stored as binary dummy columns named `genre_*`.  
We look at the 15 most common genres and how their Good-rate compares to the overall baseline.

In [ ]:
genre_cols = [c for c in X.columns if c.startswith('genre_')]

genre_stats = pd.DataFrame({
    'count'    : X[genre_cols].sum(),
    'good_rate': X[genre_cols].apply(lambda col: y[col == 1].mean())
}).sort_values('count', ascending=False).head(15)

genre_stats.index = [c.replace('genre_', '') for c in genre_stats.index]
baseline = y.mean()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: frequency
axes[0].barh(genre_stats.index[::-1], genre_stats['count'][::-1],
             color=GOOD_COLOR, alpha=0.8)
axes[0].set_xlabel('Number of Games')
axes[0].set_title('Top 15 Genres by Frequency')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

# Right: good rate vs baseline
colors_bar = [GOOD_COLOR if r >= baseline else BAD_COLOR for r in genre_stats['good_rate'][::-1]]
axes[1].barh(genre_stats.index[::-1], genre_stats['good_rate'][::-1] * 100,
             color=colors_bar, alpha=0.8)
axes[1].axvline(baseline * 100, color='black', linestyle='--', linewidth=1.2, label=f'Overall baseline ({baseline*100:.1f}%)')
axes[1].set_xlabel('Good Rate (%)')
axes[1].set_title('Good Rate by Genre (Top 15)')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

print('Good rate vs baseline by genre:')
print(genre_stats[['count', 'good_rate']].assign(vs_baseline=genre_stats['good_rate'] - baseline).round(3).to_string())

---
## Step 5 – Top Tags

Tags are user-defined labels stored as binary columns named `tag_*`.  
We kept the top 50 by frequency in preprocessing — here we visualise which tags are most associated with Good reception.

In [ ]:
tag_cols = [c for c in X.columns if c.startswith('tag_')]

tag_stats = pd.DataFrame({
    'count'    : X[tag_cols].sum(),
    'good_rate': X[tag_cols].apply(lambda col: y[col == 1].mean())
}).sort_values('good_rate', ascending=False)

tag_stats.index = [c.replace('tag_', '') for c in tag_stats.index]

top_good    = tag_stats.head(15)
bottom_good = tag_stats.tail(15)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(top_good.index[::-1], top_good['good_rate'][::-1] * 100,
             color=GOOD_COLOR, alpha=0.8)
axes[0].axvline(baseline * 100, color='black', linestyle='--', linewidth=1.2)
axes[0].set_xlabel('Good Rate (%)')
axes[0].set_title('Tags Most Associated with Good Reception')

axes[1].barh(bottom_good.index, bottom_good['good_rate'] * 100,
             color=BAD_COLOR, alpha=0.8)
axes[1].axvline(baseline * 100, color='black', linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Good Rate (%)')
axes[1].set_title('Tags Most Associated with Bad Reception')

plt.tight_layout()
plt.show()

---
## Step 6 – Platform Availability

Steam games can be released on Windows, Mac, and/or Linux.  
We check whether multi-platform availability correlates with reception quality.

In [ ]:
platform_cols = ['Windows', 'Mac', 'Linux']

platform_stats = pd.DataFrame({
    'pct_of_games': (X[platform_cols].sum() / len(X) * 100).round(1),
    'good_rate'   : X[platform_cols].apply(lambda col: y[col == 1].mean() * 100).round(1)
})
print(platform_stats)

X['n_platforms'] = X[platform_cols].sum(axis=1)

good_by_nplat = pd.Series([
    y[X['n_platforms'] == n].mean() * 100
    for n in range(4)
], index=range(4), name='good_rate')

X.drop(columns=['n_platforms'], inplace=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(platform_stats.index, platform_stats['pct_of_games'],
            color=[GOOD_COLOR, '#4a90d9', '#7db8e8'])
axes[0].set_ylabel('% of Games Available On')
axes[0].set_title('Platform Availability')
for i, v in enumerate(platform_stats['pct_of_games']):
    axes[0].text(i, v + 0.5, f'{v}%', ha='center', fontsize=10)

axes[1].bar(good_by_nplat.index, good_by_nplat.values, color=GOOD_COLOR, alpha=0.8)
axes[1].axhline(baseline * 100, color='black', linestyle='--', linewidth=1)
axes[1].set_xlabel('Number of Platforms Supported')
axes[1].set_ylabel('Good Rate (%)')
axes[1].set_title('Good Rate by Number of Supported Platforms')
axes[1].set_xticks(range(4))

plt.tight_layout()
plt.show()

---
## Step 7 – Release Era Distribution

We grouped release years into three era buckets in preprocessing: pre-2015, 2015-2019, and 2020+.  
Here we check whether era affects reception and how many games fall in each bucket.

In [ ]:
era_cols  = [c for c in X.columns if c.startswith('era_')]
era_names = [c.replace('era_', '') for c in era_cols]

era_counts    = X[era_cols].sum()
era_good_rate = X[era_cols].apply(lambda col: y[col == 1].mean() * 100)

era_summary = pd.DataFrame({
    'count'    : era_counts.values,
    'good_rate': era_good_rate.values
}, index=era_names)
print(era_summary.round(1))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(era_names, era_counts.values, color=GOOD_COLOR, alpha=0.8)
axes[0].set_ylabel('Number of Games')
axes[0].set_title('Games per Release Era')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))
for i, v in enumerate(era_counts.values):
    axes[0].text(i, v + 100, f'{v:,}', ha='center', fontsize=9)

colors = [GOOD_COLOR if r >= baseline * 100 else BAD_COLOR for r in era_good_rate.values]
axes[1].bar(era_names, era_good_rate.values, color=colors, alpha=0.8)
axes[1].axhline(baseline * 100, color='black', linestyle='--', linewidth=1,
                label=f'Overall baseline ({baseline*100:.1f}%)')
axes[1].set_ylabel('Good Rate (%)')
axes[1].set_title('Good Rate by Release Era')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Step 8 – Continuous Feature Distributions

We compare the distributions of continuous features between Good and Bad games.  
Overlapping distributions = weak signal. Separated distributions = strong signal.

**Note:** Features are stored unscaled — they will be scaled inside the modelling pipelines.

In [ ]:
plot_cols = [c for c in continuous_cols if c in X.columns]

n_cols = 3
n_rows = (len(plot_cols) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 3.5))
axes = axes.flatten()

for i, col in enumerate(plot_cols):
    ax = axes[i]
    for label, color, name in [(0, BAD_COLOR, 'Bad'), (1, GOOD_COLOR, 'Good')]:
        data = X.loc[y == label, col].dropna()
        cap  = data.quantile(0.99)
        ax.hist(data.clip(upper=cap), bins=40, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(col, fontsize=10)
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=8)

for j in range(len(plot_cols), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Continuous Feature Distributions: Good vs Bad\n(clipped at 99th percentile)',
             fontsize=12, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

---
## Step 9 – Correlation Heatmap

We compute pairwise Pearson correlations between the continuous features and the label.  
High correlations between features may indicate redundancy (multicollinearity).

In [ ]:
corr_df = X[continuous_cols].copy()
corr_df['label'] = y.values

corr_matrix = corr_df.corr()

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix, mask=mask,
    annot=True, fmt='.2f', cmap='coolwarm',
    vmin=-1, vmax=1, center=0,
    square=True, linewidths=0.5,
    ax=ax, annot_kws={'size': 8}
)
ax.set_title('Pearson Correlation: Continuous Features + Label', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

label_corr = corr_matrix['label'].drop('label').sort_values(key=abs, ascending=False)
print('Feature correlation with label (sorted by absolute value):')
print(label_corr.round(3).to_string())

---
## Step 10 – Outlier Identification (IQR Method)

An outlier is any value below Q1 - 1.5*IQR or above Q3 + 1.5*IQR.  
These are not removed — they are valid games (e.g. a game with very high playtime).  
This informs how linear vs tree-based models will handle extreme values.

In [ ]:
outlier_report = []
for col in continuous_cols:
    if col not in X.columns:
        continue
    q1  = X[col].quantile(0.25)
    q3  = X[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_out = ((X[col] < lower) | (X[col] > upper)).sum()
    outlier_report.append({
        'feature'     : col,
        'Q1'          : round(q1, 2),
        'Q3'          : round(q3, 2),
        'IQR'         : round(iqr, 2),
        'lower_bound' : round(lower, 2),
        'upper_bound' : round(upper, 2),
        'n_outliers'  : n_out,
        'pct_outliers': round(n_out / len(X) * 100, 2)
    })

outlier_df = pd.DataFrame(outlier_report).sort_values('pct_outliers', ascending=False)
print('IQR outlier analysis:')
print(outlier_df.to_string(index=False))

top_outlier_cols = outlier_df.head(6)['feature'].tolist()
fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()
for i, col in enumerate(top_outlier_cols):
    axes[i].boxplot(
        [X.loc[y == 1, col], X.loc[y == 0, col]],
        labels=['Good', 'Bad'],
        patch_artist=True,
        boxprops=dict(facecolor=GOOD_COLOR, alpha=0.5),
        medianprops=dict(color='black', linewidth=2)
    )
    axes[i].set_title(col, fontsize=10)
    axes[i].set_ylabel('Value')

plt.suptitle('Box Plots — Top 6 Features by Outlier Rate', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

---
## Step 11 – Language Support

More languages = wider audience reach. We check if games with more language support tend to score better.

In [ ]:
lang_cap = X['n_languages'].clip(upper=20)

lang_good_rate = pd.Series({
    n: y[lang_cap == n].mean() * 100
    for n in sorted(lang_cap.unique())
    if (lang_cap == n).sum() >= 50
})

lang_counts = lang_cap.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(lang_counts.index, lang_counts.values, color=GOOD_COLOR, alpha=0.8)
axes[0].set_xlabel('Number of Supported Languages (capped at 20)')
axes[0].set_ylabel('Number of Games')
axes[0].set_title('Language Support Distribution')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

axes[1].plot(lang_good_rate.index, lang_good_rate.values,
             marker='o', color=GOOD_COLOR, linewidth=2)
axes[1].axhline(baseline * 100, color='black', linestyle='--', linewidth=1,
                label=f'Baseline ({baseline*100:.1f}%)')
axes[1].set_xlabel('Number of Supported Languages (capped at 20)')
axes[1].set_ylabel('Good Rate (%)')
axes[1].set_title('Good Rate by Language Count')
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.show()

---
## Step 12 – EDA Summary

In [ ]:
print('=' * 60)
print('EDA SUMMARY')
print('=' * 60)
print(f"""
Dataset size     : {len(X):,} games x {X.shape[1]} features
Class balance    : {(y==1).mean()*100:.1f}% Good / {(y==0).mean()*100:.1f}% Bad
  Imbalance ratio approx {(y==1).sum()/(y==0).sum():.1f}:1
  Action needed: use class_weight='balanced' and/or SMOTE

Feature groups   : {len([c for c in X.columns if c.startswith('genre_')])} genre dummies
                   {len([c for c in X.columns if c.startswith('cat_')])} category dummies
                   {len([c for c in X.columns if c.startswith('tag_')])} tag dummies (top 50)
                   {len([c for c in X.columns if c.startswith('era_')])} era buckets
                   {len(continuous_cols)} continuous features (scaled inside pipelines)

Missing values   : 0 (imputed in preprocessing)
""")
print('Next step: section4a_logistic.ipynb (Zubair Ashfaq)')
print('=' * 60)

---

### Section 3 Checklist

| Analysis | Status |
|----------|--------|
| Class balance bar chart | done |
| Price distribution (free vs paid) | done |
| Top genres frequency + good rate | done |
| Top tags by good rate (best / worst) | done |
| Platform availability | done |
| Release era distribution | done |
| Continuous feature histograms (Good vs Bad) | done |
| Pearson correlation heatmap | done |
| IQR outlier analysis + box plots | done |
| Language support vs good rate | done |

**Output:** No new files are written. All modelling notebooks load from the same Parquet files.